# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [3]:
# Distribución de la variable objetivo
print(fraud["fraud"].value_counts())

# Porcentaje de cada clase
print(fraud["fraud"].value_counts(normalize=True) * 100)

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64
fraud
0.0    91.2597
1.0     8.7403
Name: proportion, dtype: float64


La variable objetivo presenta un **desequilibrio de clases**. De un total de 1.000.000 de transacciones, el **91,26 % son legítimas** y únicamente el **8,74 % son fraudulentas**.

Este desequilibrio es importante porque un modelo podría obtener una accuracy elevada prediciendo principalmente transacciones legítimas, sin detectar correctamente los fraudes.

Por ello, es recomendable evaluar el modelo utilizando métricas como **precision, recall y F1-score**, en lugar de basarnos únicamente en la accuracy.

In [4]:
# Separamos las variables predictoras y la variable objetivo
X = fraud.drop(columns=["fraud"])
y = fraud["fraud"]

# Dividimos los datos en entrenamiento (80 %) y prueba (20 %)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (800000, 7)
Prueba: (200000, 7)


In [5]:
from sklearn.linear_model import LogisticRegression

# Creamos el modelo
modelo_lr = LogisticRegression(max_iter=1000)

# Entrenamos con los datos originales
modelo_lr.fit(X_train, y_train)

# Realizamos predicciones
y_pred = modelo_lr.predict(X_test)

print("Modelo entrenado correctamente")

Modelo entrenado correctamente


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

# Evaluamos las predicciones del modelo
print("Informe de clasificación:")
print(classification_report(y_test, y_pred))

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))

Informe de clasificación:
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182519
         1.0       0.90      0.60      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000

Matriz de confusión:
[[181291   1228]
 [  6916  10565]]


El modelo de regresión logística sin balanceo obtiene una accuracy del 95,93 %, una precision del 90 % y un recall del 60 % para la clase fraudulenta.

Aunque su rendimiento global es elevado, únicamente identifica aproximadamente el 60 % de los fraudes reales, dejando sin detectar 6.916 transacciones fraudulentas.

Por tanto, la accuracy no refleja completamente el rendimiento del modelo. Sería conveniente aplicar técnicas de balanceo para intentar mejorar el recall de la clase minoritaria y comparar sus resultados.

In [7]:
from imblearn.over_sampling import RandomOverSampler

# Creamos la técnica de oversampling
ros = RandomOverSampler(random_state=42)

# Equilibramos únicamente los datos de entrenamiento
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

# Comprobamos la nueva distribución
print("Antes del balanceo:")
print(y_train.value_counts())

print("\nDespués del balanceo:")
print(y_train_over.value_counts())

Antes del balanceo:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

Después del balanceo:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


In [8]:
# Entrenamos una regresión logística con los datos balanceados
modelo_over = LogisticRegression(max_iter=1000)

modelo_over.fit(X_train_over, y_train_over)

# Predecimos sobre los datos de prueba originales
y_pred_over = modelo_over.predict(X_test)

# Evaluamos el modelo
print("Modelo con Random Oversampling")

print("\nInforme de clasificación:")
print(classification_report(y_test, y_pred_over))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_over))

Modelo con Random Oversampling

Informe de clasificación:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000


Matriz de confusión:
[[170391  12128]
 [   910  16571]]


Al aplicar Random Oversampling, el recall de la clase fraudulenta aumenta del 60 % al 95 %, reduciendo los fraudes no detectados de 6.916 a 910.

Sin embargo, la precision disminuye del 90 % al 58 %, debido al aumento de falsos positivos. El F1-score se mantiene aproximadamente en 0,72.

Por tanto, esta técnica mejora considerablemente la detección de fraudes, aunque genera más falsas alarmas. Su utilidad dependerá de si la empresa considera más costoso dejar escapar un fraude o investigar una transacción legítima.

In [9]:
from imblearn.under_sampling import RandomUnderSampler

# Creamos la técnica de undersampling
rus = RandomUnderSampler(random_state=42)

# Balanceamos únicamente los datos de entrenamiento
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("Antes del balanceo:")
print(y_train.value_counts())

print("\nDespués del balanceo:")
print(y_train_under.value_counts())

Antes del balanceo:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

Después del balanceo:
fraud
0.0    69922
1.0    69922
Name: count, dtype: int64


In [10]:
# Creamos el modelo
modelo_under = LogisticRegression(max_iter=1000)

# Entrenamos con los datos balanceados mediante Undersampling
modelo_under.fit(X_train_under, y_train_under)

# Predecimos sobre el conjunto de prueba original
y_pred_under = modelo_under.predict(X_test)

# Evaluamos el modelo
print("Modelo con Random Undersampling")

print("\nInforme de clasificación:")
print(classification_report(y_test, y_pred_under))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_under))

Modelo con Random Undersampling

Informe de clasificación:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000


Matriz de confusión:
[[170388  12131]
 [   918  16563]]


Random Undersampling consigue resultados muy similares a Random Oversampling, alcanzando un recall del 95 %, una precision del 58 % y un F1-score de 0,72 para la clase fraudulenta.

El modelo detecta 16.563 fraudes y deja escapar 918, frente a los 6.916 fraudes no detectados por el modelo sin balanceo.

Aunque mejora notablemente la detección de fraudes, también aumenta los falsos positivos. Además, Undersampling presenta la desventaja de descartar una gran cantidad de transacciones legítimas durante el entrenamiento.

In [11]:
from imblearn.over_sampling import SMOTE

# Creamos SMOTE
smote = SMOTE(random_state=42)

# Aplicamos SMOTE solo a los datos de entrenamiento
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Antes de SMOTE:")
print(y_train.value_counts())

print("\nDespués de SMOTE:")
print(y_train_smote.value_counts())

Antes de SMOTE:
fraud
0.0    730078
1.0     69922
Name: count, dtype: int64

Después de SMOTE:
fraud
0.0    730078
1.0    730078
Name: count, dtype: int64


In [12]:
# Creamos el modelo
modelo_smote = LogisticRegression(max_iter=1000)

# Entrenamos con los datos balanceados mediante SMOTE
modelo_smote.fit(X_train_smote, y_train_smote)

# Predecimos sobre el conjunto de prueba original
y_pred_smote = modelo_smote.predict(X_test)

# Evaluamos el modelo
print("Modelo con SMOTE")

print("\nInforme de clasificación:")
print(classification_report(y_test, y_pred_smote))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_smote))

Modelo con SMOTE

Informe de clasificación:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.94    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.94      0.94    200000


Matriz de confusión:
[[170498  12021]
 [   941  16540]]


### Conclusiones

El dataset presenta un desequilibrio de clases, ya que el 91,26 % de las transacciones son legítimas y únicamente el 8,74 % son fraudulentas.

Se entrenaron cuatro modelos de regresión logística: uno sin balanceo y tres aplicando Random Oversampling, Random Undersampling y SMOTE.

El modelo original obtuvo una accuracy cercana al 96 %, pero solo detectó el 60 % de los fraudes. Al aplicar técnicas de balanceo, el recall aumentó aproximadamente hasta el 95 %, reduciendo considerablemente los fraudes no detectados.

Sin embargo, esta mejora también provocó un aumento de falsos positivos y una disminución de la precision, mientras que el F1-score se mantuvo alrededor de 0,72.

Las tres técnicas de balanceo obtuvieron resultados muy similares. Random Oversampling detectó ligeramente más fraudes, aunque la diferencia no es suficiente para considerarlo claramente superior.

En conclusión, **la elección de la técnica de balanceo depende del objetivo del negocio**. Si la prioridad es detectar el mayor número posible de fraudes, las técnicas de balanceo resultan interesantes. No obstante, también debe valorarse el coste de las falsas alarmas antes de seleccionar un modelo para su aplicación real.